In [ ]:
import os
from pathlib import Path

if "pwd" not in globals():
    pwd= Path.cwd()

def running_in_kaggle() -> bool:
    return "KAGGLE_KERNEL_RUN_TYPE" in os.environ
    
print("Running in kaggle", running_in_kaggle())

In [5]:
resume_from_checkpoint_file=''

if running_in_kaggle():
    ! pip install -r /kaggle/input/datasets/solfacity/cpprl/requirements/requirements_kaggle.txt

    from pathlib import Path

    project_directory = Path("/kaggle/input/datasets/solfacity/cpprl")

    def inject_dataset():
        import sys
        sys.path.insert(0, str(project_directory))

    def create_checkpoint_archive():
        import shutil
        global resume_from_checkpoint_file
        try:
            shutil.make_archive(
                "/kaggle/working/checkpoints",
                "zip",
                "/kaggle/input/datasets/solfacity/cpprl/kaggle/checks/curve_0308000_steps",
            )

            resume_from_checkpoint_file="/kaggle/working/checkpoints"
        except:
            ...

    inject_dataset()
    create_checkpoint_archive()
    print("resume_from_checkpoint_file",resume_from_checkpoint_file)


else:
    %cd "{pwd}"
    %cd ../
    %pwd

In [6]:
from runtime.runtime_manager import (
    RuntimeConfig,
    RuntimeManager,
    read_config_url,
)

config_url = "https://api.npoint.io/f514861fbdb4b001ab3a"

runtime_manager = RuntimeManager(
    read_fn=lambda: read_config_url(
        path=config_url,
    ),
)

runtime_config = RuntimeConfig(
    render_enabled=False,
    end_training=False,
    checkpoint_save_freq=1000,
    checkpoint_save_replay_buffer=False,
    checkpoint_save_vecnormalize=False,
    csv_buffer_size=100,
    refresh_frequency=1,
)

runtime_manager.set_config(runtime_config)

In [7]:
def train_fn(manager):
    global resume_from_checkpoint_file
    from training import trainer

    trainer.total_timesteps = 8_000_000
    trainer.seed = 7
    
    trainer.resume_from_checkpoint_file=resume_from_checkpoint_file
    # trainer.env_type = "mower"
    trainer.env_type="curve"
    trainer.logger_formats = [
        "log",
        "stdout",
    ]
    trainer.start_training(
        runtime_manager=manager,
    )

In [ ]:
train_fn(runtime_manager)